In [182]:
import pypsa
import pandas as pd
from pea import Pea
from pea import preset as pp
import numpy as np

import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

pio.templates.default = 'seaborn' #https://plotly.com/python/templates/
pd.options.plotting.backend = "plotly"


%load_ext autoreload
%autoreload 2

ppv = vars(pp)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [183]:
nf45 = pypsa.Network("./result/flex_3H_2045.nc")
nf40 = pypsa.Network("./result/flex_3H_2040.nc")
nf30 = pypsa.Network("./result/flex_3H_2030.nc")

ng45 = pypsa.Network("./result/gas_3h_2045.nc")
ng40 = pypsa.Network("./result/gas_3h_2040.nc")
ng30 = pypsa.Network("./result/gas_3h_2030.nc")

Index(['FR4 0 waste CHP-2045', 'IT3 0 waste CHP-2045',
       'FR4 0 waste CHP CC-2045', 'IT3 0 waste CHP CC-2045',
       'IT3 0 urban central gas CHP-2010'],
      dtype='object', name='name')
INFO:pypsa.io:Imported network flex_3H_2045.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores
Index(['FR4 0 waste CHP-2040', 'IT3 0 waste CHP-2040',
       'FR4 0 waste CHP CC-2040', 'IT3 0 waste CHP CC-2040',
       'IT3 0 urban central gas CHP-2005', 'IT3 0 urban central gas CHP-2010'],
      dtype='object', name='name')
INFO:pypsa.io:Imported network flex_3H_2040.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores
INFO:pypsa.io:Imported network flex_3H_2030.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores
Index(['FR4 0 waste CHP-2045', 'IT3 0 waste CHP-2045',
       'FR4 0 waste CHP CC-2045', 'IT3 0 waste CHP CC-2045',
       'IT3 0 urban central g

In [184]:
pf45 = Pea(nf45, config = {"resolution": 3})
pf40 = Pea(nf40, config = {"resolution": 3})
pf30 = Pea(nf30, config = {"resolution": 3})

pg45 = Pea(ng45, config = {"resolution": 3})
pg40 = Pea(ng40, config = {"resolution": 3})
pg30 = Pea(ng30, config = {"resolution": 3})

peas = {
  'gas 2030': pg30,
  'flex 2030': pf30,
  'gas 2040': pg40,
  'flex 2040': pf40,
  'gas 2045': pg45,
  'flex 2045': pf45
}

def fig_total (df, fig):
  total_s = df.sum(axis=1).round(2)
  fig.add_trace(go.Scatter(
    x=total_s.index, 
    y=total_s,
    text=total_s,
    mode='text',
    textposition='top center',
    showlegend=False
))

pg_keys = [
        'c_pg_onwind', 'c_pg_offwind', 'c_pg_pv',
        'c_pg_biomass', 'c_pg_waste', 'c_pg_water',
        'c_pg_battery', 'c_pg_phs', 'c_pg_dsm',
        'c_pg_import', 
        'c_pg_natgas', 'c_pg_h2', 'c_pg_retrofit_h2',
        ]

pc_keys = [
  'c_pc_agriculture', 'c_pc_industry', 'c_pc_ghd',
  'c_pc_electrolysis', 'c_pc_bev_charger',
  'c_pc_resistive', 'c_pc_dac', 'c_pc_heat_pump',
   'c_pc_dsm', 'c_pc_phs', 'c_pc_battery',
  'c_pc_export',
]

e_keys = ['c_e_battery',  'c_e_DSM', 'c_e_PHS']

p1_keys = ['c_pg_natgas','c_pg_h2','c_pg_retrofit_h2', 'c_pg_waste', 'c_pg_dsm', 'c_pg_battery', 'c_pg_biomass',
             'c_pc_export', 'c_pg_import',
             'c_pc_heat_pump', 'c_pc_dac', 'c_pc_resistive', 'c_pc_dsm', 'c_pc_battery', 'c_pc_bev_charger', 'c_pc_electrolysis'
             ]

In [185]:
import plotly.graph_objects as go

capital_df = pd.DataFrame()
for nkey, pea in peas.items():
  capital_s = pd.Series(0.00, index=pg_keys)
  for key in pg_keys:
    c_key = key
    if key == 'c_pg_battery': 
      capital_s[key]  =  pea.get(['battery', 'home battery', 'battery discharger', 'home battery discharger', 'battery charger', 'home battery charge']).capital_cost() / 1e9
    elif key == 'c_pg_import':
      capital_s[key]  =  pea.get(ppv[c_key], type='import').capital_cost() / 1e9
    else:
      capital_s[key] = pea.get(ppv[c_key]).capital_cost() / 1e9

  capital_s = capital_s.rename(index= pp.map_name)
  
  capital_df[nkey] = capital_s


capital_df = capital_df.T
color_map = [pp.map_color[i] for i in pg_keys]



fig = px.bar(capital_df, color_discrete_map = pp.map_name_color, 
       title = "Power Generation CAPEX coset"
       )
fig_total(capital_df, fig)
fig.update_xaxes(title_text='Scenario')
fig.update_yaxes(title_text='Capital COST in billion EUR')
fig.show()

fig_capex = fig

In [186]:
capital_df = pd.DataFrame()
for nkey, pea in peas.items():
  capital_s = pd.Series(0.00, index=pg_keys)
  for key in pg_keys:
    c_key = key

    p_unit = 'p'
    if key in ['c_pg_natgas', 'c_pg_retrofit_h2', 'c_pg_water', 'c_pg_dsm', 'c_pg_battery', 'c_pg_biomass']:
      p_unit = 'p1'

    if key == 'c_pg_import':
      capital_s[key]  =  pea.get(ppv[c_key], type='import').opex_cost(p_unit) / 1e9
    else:
      capital_s[key]  = pea.get(ppv[c_key]).opex_cost(p_unit) / 1e9

  capital_s = capital_s.rename(index= pp.map_name)
  
  capital_df[nkey] = capital_s


capital_df = capital_df.T
color_map = [pp.map_color[i] for i in pg_keys]


fig = px.bar(capital_df, color_discrete_map = pp.map_name_color, 
       title = "Power Generation marginal cost"
)
fig_total(capital_df, fig)

fig.update_xaxes(title_text='Scenario')
fig.update_yaxes(title_text='marginal cost in billion EUR')

fig_marginal = fig

In [187]:
capital_df = pd.DataFrame()
for nkey, pea in peas.items():
  capital_s = pd.Series(0.00, index=pg_keys)
  for key in pg_keys:
    c_key = key

    p_unit = 'p'
    if key in ['c_pg_natgas', 'c_pg_retrofit_h2', 'c_pg_water', 'c_pg_dsm', 'c_pg_battery', 'c_pg_biomass']:
      p_unit = 'p1'

    if key == 'c_pg_import':
      capital_s[key]  =  pea.get(ppv[c_key], type='import').total_cost(p_unit) / 1e9
    else:
      capital_s[key]  = pea.get(ppv[c_key]).total_cost(p_unit) / 1e9
  capital_s = capital_s.rename(index= pp.map_name)
  
  capital_df[nkey] = capital_s


capital_df = capital_df.T
color_map = [pp.map_color[i] for i in pg_keys]


fig = px.bar(capital_df, color_discrete_map = pp.map_name_color, 
       title = "Power Generation total coset")
fig_total(capital_df, fig)
fig.update_xaxes(title_text='Scenario')
fig.update_yaxes(title_text='total cost in billion EUR')

fig_total_cost = fig

In [188]:
import plotly.graph_objects as go

capital_df = pd.DataFrame()
for nkey, pea in peas.items():
  capital_s = pd.Series(0.00, index=pg_keys)
  for key in pg_keys:
    c_key = key
    
    if key == 'c_pg_import':
      capital_s[key]  =  pea.get(ppv[c_key], type='import').p_nom_opt() / 1e3
    else:
      capital_s[key] = pea.get(ppv[c_key]).p_nom_opt() / 1e3

  capital_s = capital_s.rename(index= pp.map_name)
  capital_df[nkey] = capital_s

capital_df = capital_df.T
color_map = [pp.map_color[i] for i in pg_keys]

fig = px.bar(capital_df, color_discrete_map = pp.map_name_color, 
       title = "Installed Capacity in Germany",
       )
fig_total(capital_df, fig)
fig.update_xaxes(title_text='Scenario')
fig.update_yaxes(title_text='installed capacity in GW')
fig.show()

fig_pg_gw = fig

In [189]:
capital_df = pd.DataFrame()
for nkey, pea in peas.items():
  capital_s = pd.Series(0.00, index=pc_keys)
  for key in pc_keys[::-1]:
    c_key = key
    
    if key == 'c_pc_export':
      capital_s[key]  =  pea.get(ppv[c_key], type='export').p_nom_opt() / 1e3
    else:
      capital_s[key] = pea.get(ppv[c_key]).p_nom_opt() / 1e3

  capital_s = capital_s.rename(index= pp.map_name)
  capital_df[nkey] = capital_s

capital_df = capital_df.T
color_map = [pp.map_color[i] for i in pc_keys]

fig = px.bar(capital_df, color_discrete_map = pp.map_name_color, 
       title = "power consumption in Germany",
       )
fig_total(capital_df, fig)
fig.update_xaxes(title_text='Scenario')
fig.update_yaxes(title_text='power in GW')
fig.show()

fig_pc_gw = fig

In [190]:
capital_df = pd.DataFrame()
for nkey, pea in peas.items():
  capital_s = pd.Series(0.00, index=e_keys)
  for key in e_keys:
    
    if key == 'c_e_PHS':
      e = pea.get(ppv[key]).df['p_nom'] * pea.get(ppv[key]).df['max_hours']
      capital_s[key] = e.sum() / 1e3

    else:
      capital_s[key] = pea.get(ppv[key]).e_nom_opt() / 1e3

  capital_s = capital_s.rename(index= pp.map_name)
  capital_df[nkey] = capital_s

capital_df = capital_df.T
color_map = [pp.map_color[i] for i in e_keys]

fig = px.bar(capital_df, color_discrete_map = pp.map_name_color, 
       title = "Storage capacity in Germany",
       )
fig_total(capital_df, fig)
fig.update_xaxes(title_text='Scenario')
fig.update_yaxes(title_text='capacity in GWh')
fig.show()

fig_pe_gwh = fig

In [191]:
capital_df = pd.DataFrame()
co2_store_0 = ['co2 sequestered']
co2_store_2 = ['Sabatier', 'biogas to gas CC', 'Fischer-Tropsch', 'process emissions CC', ]
co2_store_3 = ['SMR CC', 'biomass to liquid CC', 'solid biomass for industry CC', 'gas for industry CC', 'methanolisation', 'DAC']
co2_store_4 = ['urban central gas CHP CC', 'urban central solid biomass CHP CC', 'waste CHP CC',]

for nkey, pea in peas.items():
  capital_s = pd.Series()

  for key in co2_store_0:
    p = - pea.get(key).t('p0') # * pea.get(key).df['efficiency'] 
    capital_s[key] = p.sum().sum() * 3 / 1e6

  for key in co2_store_2:
    p = - pea.get(key).t('p2') # * pea.get(key).df['efficiency2'] 
    capital_s[key] = p.sum().sum() * 3 / 1e6
  
  for key in co2_store_3:
   
    p = - pea.get(key).t('p3') #* pea.get(key).df['efficiency3'] 
    capital_s[key] = p.sum().sum() * 3 / 1e6

  for key in co2_store_4:
    p = - pea.get(key).t('p4') #* pea.get(key).df['efficiency4'] 
    capital_s[key] = p.sum().sum() * 3 / 1e6


  # capital_s = capital_s.rename(index= pp.map_name)
  capital_df[nkey] = capital_s

capital_df = capital_df.T
color_map = [pp.map_color[i] for i in e_keys]

fig = px.bar(capital_df,  
       title = "CO2 CC",
       )
fig.update_xaxes(title_text='Scenario')
fig.update_yaxes(title_text='MtCO2')
fig_co2_s = fig
# fig.show()

In [192]:
capital_df = pd.DataFrame()
co2_atmosphere_1 = ['HVC to air', 'process emissions' ] 
co2_atmosphere_2 = ['OCGT', 'SMR CC', 'SMR', 'urban central gas boiler', 'biogas to gas', 'biomass to liquid', 'biomass to liquid CC', 'solid biomass for industry CC',
                     'gas for industry CC',    'agriculture machinery oil', 'DAC',
                    'rural gas boiler', 'urban decentral gas boiler', 'oil', 'CCGT', 'lignite', 'gas compressing', 'shipping methanol', 'industry methanol', 'oil refining'
                    'land transport oil', 'gas for industry','shipping oil','kerosene for aviation','coal for industry',
                    'nuclear',
                    ] 
co2_atmosphere_3 = ['urban central gas CHP', 'urban central gas CHP CC', 'biogas to gas CC', 'urban central solid biomass CHP CC', 'waste CHP', 'waste CHP CC', 
                    'urban central gas CHP', 'urban central gas CHP CC', ]

for nkey, pea in peas.items():
  capital_s = pd.Series()

  # for key in co2_store_0:
  #   p = - pea.get(key).t('p0') * pea.get(key).df['efficiency'] 
  #   capital_s[key] = p.sum().sum() * 3 / 1e6

  for key in co2_atmosphere_1:
    p = - pea.get(key).t('p1') #* pea.get(key).df['efficiency'] 
    capital_s[key] = p.sum().sum() * 3 / 1e6

  for key in co2_atmosphere_2:
    if (pea.get(key) != None) and (pea.get(key).df is not None):
      p = - pea.get(key).t('p2')# * pea.get(key).df['efficiency2'] 
      capital_s[key] = p.sum().sum() * 3 / 1e6
  
  for key in co2_atmosphere_3:
   
    p = - pea.get(key).t('p3')# * pea.get(key).df['efficiency3'] 
    capital_s[key] = p.sum().sum() * 3 / 1e6

  # for key in co2_atmosphere_4:
  #   p = - pea.get(key).t('p4') * pea.get(key).df['efficiency4'] 
  #   capital_s[key] = p.sum().sum() * 3 / 1e6


  # capital_s = capital_s.rename(index= pp.map_name)
  capital_df[nkey] = capital_s

capital_df = capital_df.T
color_map = [pp.map_color[i] for i in e_keys]

fig = px.bar(capital_df,  
       title = "CO2 atmosphere",
       )
fig.update_xaxes(title_text='Scenario')
fig.update_yaxes(title_text='MtCO2')
fig_co2_atmosphere = fig
# fig.show()

In [193]:
pea.get('gas compressing').df == None

True

In [194]:
nf45.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,location,underwater_fraction,overnight_cost,reversed,voltage,under_construction,retrofitted,dc,fixed_profile_scaling_opt,tags
Link,,,,,,,,,,,,,,,,,,,,,
relation/10377412-320-DC,FR0 0,GB2 0,,DC,0.957698,True,0,inf,1000.0,0.0,...,,0.888446,3.324330e+06,False,320.0,0.0,NaN,1.0,0.0,relation/10377412
relation/13295785-515-DC,NO1 0,GB2 0,,DC,0.948235,True,0,inf,1400.0,0.0,...,,0.983350,4.767363e+06,False,515.0,0.0,NaN,1.0,0.0,relation/13295785
relation/14126301-450-DC,GB2 0,NL0 0,,DC,0.965761,True,0,inf,1000.0,0.0,...,,0.977109,2.441674e+06,False,450.0,0.0,NaN,1.0,0.0,relation/14126301
relation/15772117-320-DC,GB2 0,FR0 0,,DC,0.957698,True,0,inf,1000.0,0.0,...,,0.714775,2.983623e+06,False,320.0,0.0,NaN,1.0,0.0,relation/15772117
relation/15781671-525-DC,GB2 0,DK0 0,,DC,0.957469,True,0,inf,1400.0,0.0,...,,0.816407,3.209938e+06,False,525.0,0.0,NaN,1.0,0.0,relation/15781671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DE0 0 OCGT-2045,EU gas,DE0 0,,OCGT,0.400000,True,0,inf,25000.0,0.0,...,,NaN,NaN,False,NaN,NaN,NaN,NaN,0.0,
DE0 0 urban-central-gas-CHP-2045,EU gas,DE0 0,,urban central gas CHP,0.350000,True,0,inf,20000.0,0.0,...,,NaN,NaN,False,NaN,NaN,NaN,NaN,0.0,
DE0 0 urban-central-gas-CHP-CC-2045,EU gas,DE0 0,,urban central gas CHP CC,0.400000,True,0,inf,5000.0,0.0,...,,NaN,NaN,False,NaN,NaN,NaN,NaN,0.0,


In [195]:
capital_df = pd.DataFrame()
for nkey, pea in peas.items():
  capital_s = pd.Series(0.00, index=pg_keys + pc_keys)

  for key in pg_keys:
    p_unit = 'p'
    if key in p1_keys:
      p_unit = 'p1'

    if key == 'c_pg_import':
      capital_s[key]  =  pea.get(ppv[key], type='import').energy(p_unit) / 1e6
    else:
      capital_s[key] = pea.get(ppv[key]).energy(p_unit) / 1e6

  for key in pc_keys:
    p_unit = 'p'
    if key in p1_keys:
      p_unit = 'p1'

    if key == 'c_pc_export':
      capital_s[key]  =  - pea.get(ppv[key], type='export').energy(p_unit) / 1e6
    else:
      capital_s[key] = - pea.get(ppv[key]).energy(p_unit) / 1e6

  capital_s = capital_s.rename(index= pp.map_name)
  capital_df[nkey] = capital_s

capital_df = capital_df.T
color_map = [pp.map_color[i] for i in pg_keys + pc_keys]

fig = px.bar(capital_df, color_discrete_map = pp.map_name_color, 
       title = "power balance in Germany", 
       )
fig.update_xaxes(title_text='Scenario')
fig.update_yaxes(title_text='power in TWh')

fig.update_layout(
    height=800 
)
fig.show()


fig_balance_total = fig


# Power balance

In [196]:

def get_balance_t (pea):
  flex_df = pd.DataFrame()
  for key in pg_keys:
    p_unit = 'p'
    if key in p1_keys:
      p_unit = 'p1'

    if key == 'c_pg_import':
      flex_df[key]  =  pea.get(ppv[key], type='import').p(p_unit) / 1e6
    else:
      flex_df[key] = pea.get(ppv[key]).p(p_unit) / 1e6

  for key in pc_keys:
    p_unit = 'p'
    if key in p1_keys:
      p_unit = 'p1'

    if key == 'c_pc_export':
      flex_df[key]  =  - pea.get(ppv[key], type='export').p(p_unit) / 1e6
    else:
      flex_df[key] = - pea.get(ppv[key]).p(p_unit)  / 1e6

  flex_df = flex_df.resample('ME').sum()
  flex_df.rename(columns=pp.map_name, inplace=True)
  return flex_df

def plot_balance_t (df , year= '2045'):
  df = df.T
  fig = px.bar(df, color_discrete_map = pp.map_name_color, 
        title = f"power balance in Germany {year}",
        )
  fig.update_xaxes(title_text='Scenario and month')
  fig.update_yaxes(title_text='power in TWh')

  fig.update_layout(
      height=800 
  )
  return fig


def balance_data_rename (df, name):
  index_mapping = {
    '2019-01-31': f'{name}1',
    '2019-02-28': f'{name}2',
    '2019-03-31': f'{name}3',
    '2019-04-30': f'{name}4',
    '2019-05-31': f'{name}5',
    '2019-06-30': f'{name}6',
    '2019-07-31': f'{name}7',
    '2019-08-31': f'{name}8',
    '2019-09-30': f'{name}9',
    '2019-10-31': f'{name}10',
    '2019-11-30': f'{name}11',
    '2019-12-31': f'{name}12'
  } 
  df.index = df.index.strftime('%Y-%m-%d')
  df = df.rename(index=index_mapping)
  return df

def get_cross_df (gas_df, flex_df):
  gas_df['temp_index'] = [int(i.split()[-1]) for i in gas_df.index]
  flex_df['temp_index'] = [int(i.split()[-1]) for i in flex_df.index]


  gas_df['new_index'] = gas_df['temp_index'] * 2
  flex_df['new_index'] = flex_df['temp_index'] * 2 + 1

  gas_df.set_index('new_index', inplace=True)
  flex_df.set_index('new_index', inplace=True)

  gas_df.drop('temp_index', axis=1, inplace=True)
  flex_df.drop('temp_index', axis=1, inplace=True)


  result = pd.concat([gas_df, flex_df]).sort_index()

  original_index = [f'gas {i}' if i % 2 == 0 else f'flex {i//2 + 1}' for i in range(24)]
  result.index = original_index
  return result

def get_month_df (gas_df, flex_df):
  df = pd.DataFrame({})

  for month in range(1, 13):
    month_str = str(month)

    df[f'gas{month_str}'] = gas_df.loc[month_str]
    df[f'flex{month_str}'] = flex_df.loc[month_str]

    df[month_str] = pd.Series(0, index=pg_keys + pc_keys)
  return df


f45_df = get_balance_t(pf45)
f45_df = balance_data_rename(f45_df, '')

g45_df = get_balance_t(pg45)
g45_df = balance_data_rename(g45_df, '')

df45 = get_month_df(g45_df, f45_df)
fig_balance_45 = plot_balance_t(df45, year= '2045')

f40_df = get_balance_t(pf40)
f40_df = balance_data_rename(f40_df, '')

g40_df = get_balance_t(pg40)
g40_df = balance_data_rename(g40_df, '')

df40 = get_month_df(g40_df, f40_df)
fig_balance_40 = plot_balance_t(df40, year= '2040')

f30_df = get_balance_t(pf30)
f30_df = balance_data_rename(f30_df, '')

g30_df = get_balance_t(pg30)
g30_df = balance_data_rename(g30_df, '')

df30 = get_month_df(g30_df, f30_df)
fig_balance_30 = plot_balance_t(df30, year= '2030')


In [197]:

def plot_month_balance (pea, start = '2019-1-1 00:00:00', end = '2019-1-30 00:00:00', title= "01.2045"):
  flex_df = pd.DataFrame()
  for key in pg_keys:
    p_unit = 'p'
    if key in p1_keys:
      p_unit = 'p1'

    if key == 'c_pg_import':
      flex_df[key]  =  pea.get(ppv[key], type='import').p(p_unit) / 1e6
    else:
      flex_df[key] = pea.get(ppv[key]).p(p_unit) / 1e6

  for key in pc_keys:
    p_unit = 'p'
    if key in p1_keys:
      p_unit = 'p1'

    if key == 'c_pc_export':
      flex_df[key]  =  - pea.get(ppv[key], type='export').p(p_unit) / 1e6
    else:
      flex_df[key] = - pea.get(ppv[key]).p(p_unit)  / 1e6

  flex_df = flex_df[start: end]
  flex_df.rename(columns=pp.map_name, inplace=True)

  fig = px.bar(flex_df, color_discrete_map = pp.map_name_color, 
          title = f"power balance in Germany {title}",
          )
  fig.update_xaxes(title_text='Scenario and month')
  fig.update_yaxes(title_text='power in TWh')

  fig.update_layout(
      height=800 
  )
  return fig

fig_f_1_45 = plot_month_balance(pf45, title = '01.2045 flex')
fig_g_1_45 = plot_month_balance(pg45, title = '01.2045 gas')

fig_f_6_45 = plot_month_balance(pf45,  start = '2019-6-1 00:00:00', end = '2019-6-30 00:00:00', title = '06.2045 flex')
fig_g_6_45 = plot_month_balance(pg45, start = '2019-6-1 00:00:00', end = '2019-6-30 00:00:00',title = '06.2045 gas')

fig_f_4_45 = plot_month_balance(pf45,  start = '2019-4-1 00:00:00', end = '2019-4-30 00:00:00', title = '04.2045 flex')
fig_g_4_45 = plot_month_balance(pg45, start = '2019-4-1 00:00:00', end = '2019-4-30 00:00:00',title = '04.2045 gas')

In [198]:
from plotly.subplots import make_subplots

combined_fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("CAPEX", "OPEX", "Total")
)

for trace in fig_capex.data:
    combined_fig.add_trace(trace, row=1, col=1)

for trace in fig_marginal.data:
    combined_fig.add_trace(trace, row=1, col=2)

for trace in fig_total_cost.data:
    combined_fig.add_trace(trace, row=1, col=3, )

combined_fig.update_layout(title_text='Cost in Germany [billion EUR]', barmode='stack', height=800 )

combined_fig.show()

In [199]:
from plotly.subplots import make_subplots

combined_fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Supply", "Demand", "Total")
)

for trace in fig_pg_gw.data:
    combined_fig.add_trace(trace, row=1, col=1)

for trace in fig_pc_gw.data:
    combined_fig.add_trace(trace, row=1, col=2, )

combined_fig.update_layout(title_text='Capacity Germany [GW]', barmode='stack', height=800 )

combined_fig.show()

In [200]:
fig_balance_total.update_layout(
  height=800,
  width = 500
)

In [201]:
fig_balance_45.show()
fig_balance_40.show()
fig_balance_30.show()

In [202]:
fig_f_1_45.show()
fig_g_1_45.show()

fig_f_4_45.show()
fig_g_4_45.show()


fig_f_6_45.show()
fig_g_6_45.show()

In [203]:
fig_co2_s.show()
fig_co2_atmosphere.show()